# CTR Prediction Pipeline

Objective: minimize **Normalized Cross-Entropy (NCE)** on test set (week 4).

```
NCE = LogLoss(y, ŷ) / LogLoss(y, p̄)    # lower is better; < 1.0 beats naïve baseline
```

Steps: load → prepare → features → baseline → importance gate → tune → calibrate → submit

## 0 · Imports

In [ ]:
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder

optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = Path("../the-ad-ecosystem-and-ctr-prediction")
SUB_DIR  = Path("../submissions")
SUB_DIR.mkdir(exist_ok=True)

## 1 · Load Data

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")

print(f"train: {train.shape}  |  test: {test.shape}")
print(f"train CTR (p̄): {train['clicked'].mean():.4f}")

In [ ]:
train.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=0.9)
_eda = train.copy()
_eda["clicked_int"] = _eda["clicked"].astype(int)
_eda["timestamp"]   = pd.to_datetime(_eda["timestamp"])
_eda["date"]        = _eda["timestamp"].dt.date

fig, axes = plt.subplots(4, 3, figsize=(16, 20))
fig.suptitle("EDA — CTR Prediction Dataset", fontsize=14, y=1.01)

# ── 1. Target distribution ────────────────────────────────────────────────────
ax = axes[0, 0]
vc = _eda["clicked_int"].value_counts()
sns.barplot(x=["Not Clicked (0)", "Clicked (1)"], y=vc.values, ax=ax, palette=["#5b8db8", "#e07b54"])
ax.set_title("Target distribution")
ax.set_ylabel("Count")
for bar, v in zip(ax.patches, vc.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2000,
            f"{v:,}\n({v/len(_eda)*100:.1f}%)", ha="center", va="bottom", fontsize=8)

# ── 2. Daily CTR over time ────────────────────────────────────────────────────
ax = axes[0, 1]
daily = _eda.groupby("date")["clicked_int"].mean().reset_index()
ax.plot(daily["date"], daily["clicked_int"], marker="o", markersize=3, linewidth=1.5, color="#5b8db8")
ax.axhline(daily["clicked_int"].mean(), color="#e07b54", linestyle="--", linewidth=1, label=f"mean={daily['clicked_int'].mean():.4f}")
ax.set_title("Daily CTR (train)")
ax.set_xlabel("")
ax.set_ylabel("CTR")
ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=8)

# ── 3. Hourly CTR ─────────────────────────────────────────────────────────────
ax = axes[0, 2]
hourly = _eda.groupby("hour")["clicked_int"].mean().reset_index()
sns.barplot(data=hourly, x="hour", y="clicked_int", ax=ax, color="#5b8db8")
ax.axhline(_eda["clicked_int"].mean(), color="#e07b54", linestyle="--", linewidth=1)
ax.set_title("CTR by hour")
ax.set_xlabel("Hour")
ax.set_ylabel("CTR")

# ── 4. CTR by creative_size ───────────────────────────────────────────────────
ax = axes[1, 0]
cs = _eda.groupby("creative_size")["clicked_int"].mean().sort_values(ascending=False).reset_index()
sns.barplot(data=cs, x="clicked_int", y="creative_size", ax=ax, palette="Blues_r")
ax.axvline(_eda["clicked_int"].mean(), color="#e07b54", linestyle="--", linewidth=1)
ax.set_title("CTR by creative_size")
ax.set_xlabel("CTR")
ax.set_ylabel("")

# ── 5. CTR by banner_pos ──────────────────────────────────────────────────────
ax = axes[1, 1]
bp = _eda.groupby("banner_pos")["clicked_int"].mean().reset_index()
sns.barplot(data=bp, x="banner_pos", y="clicked_int", ax=ax, color="#5b8db8")
ax.axhline(_eda["clicked_int"].mean(), color="#e07b54", linestyle="--", linewidth=1)
ax.set_title("CTR by banner_pos")
ax.set_xlabel("Banner position")
ax.set_ylabel("CTR")

# ── 6. CTR by device_type / is_app ───────────────────────────────────────────
ax = axes[1, 2]
dev = _eda.groupby("device_type")["clicked_int"].mean().sort_values(ascending=False).reset_index()
app = _eda.groupby("is_app")["clicked_int"].mean().reset_index()
app["device_type"] = app["is_app"].map({True: "App", False: "Web"})
combined = pd.concat([dev, app[["device_type", "clicked_int"]]], ignore_index=True)
sns.barplot(data=combined, x="device_type", y="clicked_int", ax=ax,
            palette=["#5b8db8", "#7eb5d6", "#a8cfe0", "#e07b54", "#f0a882"])
ax.axhline(_eda["clicked_int"].mean(), color="black", linestyle="--", linewidth=1)
ax.set_title("CTR: device_type & channel")
ax.set_xlabel("")
ax.set_ylabel("CTR")
ax.tick_params(axis="x", rotation=20)

# ── 7. CTR by user_segment ────────────────────────────────────────────────────
ax = axes[2, 0]
seg = _eda.groupby("user_segment")["clicked_int"].mean().sort_values(ascending=False).reset_index()
sns.barplot(data=seg, x="clicked_int", y="user_segment", ax=ax, palette="Blues_r")
ax.axvline(_eda["clicked_int"].mean(), color="#e07b54", linestyle="--", linewidth=1)
ax.set_title("CTR by user_segment")
ax.set_xlabel("CTR")
ax.set_ylabel("")

# ── 8. CTR by day_of_week ─────────────────────────────────────────────────────
ax = axes[2, 1]
dow_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow = _eda.groupby("day_of_week")["clicked_int"].mean().reindex(dow_order).reset_index()
sns.barplot(data=dow, x="day_of_week", y="clicked_int", ax=ax,
            palette=["#5b8db8"]*5 + ["#e07b54"]*2)
ax.axhline(_eda["clicked_int"].mean(), color="black", linestyle="--", linewidth=1)
ax.set_title("CTR by day_of_week")
ax.set_xlabel("")
ax.set_ylabel("CTR")

# ── 9. CTR by device_conn_type ────────────────────────────────────────────────
ax = axes[2, 2]
conn = _eda.groupby("device_conn_type")["clicked_int"].mean().sort_values(ascending=False).reset_index()
sns.barplot(data=conn, x="clicked_int", y="device_conn_type", ax=ax, palette="Blues_r")
ax.axvline(_eda["clicked_int"].mean(), color="#e07b54", linestyle="--", linewidth=1)
ax.set_title("CTR by device_conn_type")
ax.set_xlabel("CTR")
ax.set_ylabel("")

# ── 10. historical_user_ctr distribution ─────────────────────────────────────
ax = axes[3, 0]
sns.histplot(_eda["historical_user_ctr"], bins=50, ax=ax, color="#5b8db8", kde=True)
ax.set_title("historical_user_ctr distribution")
ax.set_xlabel("historical_user_ctr")

# ── 11. Numeric correlations with target ──────────────────────────────────────
ax = axes[3, 1]
num_cols = ["hour", "banner_pos", "ad_quality_score", "C1", "C15", "C16", "C21",
            "user_depth", "historical_user_ctr"]
_corr_df = _eda[num_cols + ["is_app", "clicked_int"]].copy()
_corr_df["is_app"] = _corr_df["is_app"].astype(int)
corrs = _corr_df.corr()["clicked_int"].drop("clicked_int").sort_values()
colors = ["#e07b54" if v < 0 else "#5b8db8" for v in corrs.values]
ax.barh(corrs.index, corrs.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Pearson corr with clicked")
ax.set_xlabel("Correlation")

# ── 12. creative_size × banner_pos heatmap (CTR) ─────────────────────────────
ax = axes[3, 2]
pivot = _eda.groupby(["creative_size", "banner_pos"])["clicked_int"].mean().unstack()
sns.heatmap(pivot, ax=ax, cmap="YlOrRd", annot=True, fmt=".2f", linewidths=0.5,
            cbar_kws={"shrink": 0.8})
ax.set_title("CTR: creative_size × banner_pos")
ax.set_xlabel("banner_pos")
ax.set_ylabel("")

plt.tight_layout()
plt.show()

## 2 · Prepare

In [ ]:
def prepare(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # timestamps
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # structural NaNs — NULLs carry semantic meaning, do not impute as 'unknown'
    df.loc[df["is_app"] == True,  ["site_id", "site_domain"]] = "_app"
    df.loc[df["is_app"] == False, ["app_id",  "app_domain"]]  = "_site"

    return df


train = prepare(train)
test  = prepare(test)

## 3 · Feature Engineering Variants

Three self-contained variants — pick one by setting `FEATURE_VARIANT` in §5.

| Variant | What it adds | When to use |
|---|---|---|
| `"simple"` | Raw columns + label-encoding only | Fastest sanity-check baseline |
| `"standard"` | + temporal flags + target encoding + count features | Default starting point |
| `"advanced"` | + interaction features + cross target encodings + user rolling CTR | After standard is solid |
| `"special"` | Unified pipeline: temporal + interactions + Bayesian CTR + log-freq + label encoding | Full-featured, clean implementation |

## 4 · Validation Split & Metric

In [ ]:
# Time-based split: train on weeks 1–2, validate on week 3
# Never use random k-fold — test is a future time window

VAL_START = pd.Timestamp("2024-01-15")

mask_tr  = train["timestamp"] < VAL_START
mask_val = train["timestamp"] >= VAL_START

print(f"train fold : {mask_tr.sum():>7,} rows  ({mask_tr.mean():.1%})")
print(f"val   fold : {mask_val.sum():>7,} rows  ({mask_val.mean():.1%})")


def nce(y_true: np.ndarray, y_pred: np.ndarray, base_ctr: float | None = None) -> float:
    """Normalized Cross-Entropy. Lower is better; < 1.0 beats naïve baseline."""
    if base_ctr is None:
        base_ctr = float(y_true.mean())
    baseline_ll = log_loss(y_true, np.full(len(y_true), base_ctr))
    return log_loss(y_true, y_pred) / baseline_ll

## 5 · Build Feature Matrix

Set `FEATURE_VARIANT` to `"simple"`, `"standard"`, or `"advanced"`.

In [ ]:
ID_COL    = "impression_id"
LABEL     = "clicked"
SMOOTH_K  = 10
BASE_CTR  = train[LABEL].mean()

# ── Special variant config ────────────────────────────────────────────────────

# String/object categorical columns (will be label-encoded to int32)
OBJ_CAT_COLS = [
    'publisher_id', 'ad_id', 'ad_campaign_id',
    'site_id', 'app_id', 'site_domain', 'app_domain',
    'site_category', 'device_type', 'device_model',
    'device_conn_type', 'creative_size', 'user_segment', 'day_of_week',
]
# Integer columns that are actually categorical (small unique count)
INT_AS_CAT = ['C1', 'C15', 'C16', 'C21', 'banner_pos']

# Columns for Bayesian CTR + log-frequency stats
# Computed from TRAINING FOLD ONLY to prevent leakage into val/test
STAT_COLS = [
    'publisher_id', 'ad_id', 'ad_campaign_id', 'user_id',
    'site_id', 'app_id', 'site_category',
    'device_type', 'device_model', 'device_conn_type',
    'creative_size', 'banner_pos', 'hour', 'day_of_week',
    'C1', 'C15', 'C16', 'C21', 'user_segment', 'time_of_day',
    'pub_x_is_app', 'pub_x_device', 'site_cat_x_device', 'campaign_x_device',
]
# Columns to remove from final feature matrix
_SPECIAL_DROP = [ID_COL, 'user_id']  # user_id replaced by user_id_ctr


# ── 1. Temporal features ──────────────────────────────────────────────────────
def _add_temporal(df):
    df['is_weekend'] = df['day_of_week'].isin(['Sat', 'Sun']).astype(int)
    df['hour_sin']   = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['hour'] / 24)
    cond = [
        (df['hour'] >= 6)  & (df['hour'] < 12),
        (df['hour'] >= 12) & (df['hour'] < 17),
        (df['hour'] >= 17) & (df['hour'] < 22),
    ]
    df['time_of_day'] = np.select(cond, ['morning', 'afternoon', 'evening'], default='night')
    dow = {'Mon': 0, 'Tue': 1, 'Wed': 2, 'Thu': 3, 'Fri': 4, 'Sat': 5, 'Sun': 6}
    dn  = df['day_of_week'].map(dow)
    df['dow_sin'] = np.sin(2 * np.pi * dn / 7)
    df['dow_cos'] = np.cos(2 * np.pi * dn / 7)
    return df.drop(columns=['timestamp'])


# ── 2. Interaction features ───────────────────────────────────────────────────
def _add_interactions(df):
    s = lambda c: df[c].fillna('__NULL__').astype(str)
    df['pub_x_is_app']      = s('publisher_id')   + '__' + s('is_app')
    df['pub_x_device']      = s('publisher_id')   + '__' + s('device_type')
    df['site_cat_x_device'] = s('site_category')  + '__' + s('device_type')
    df['banner_x_device']   = s('banner_pos')     + '__' + s('device_type')
    df['creative_x_banner'] = s('creative_size')  + '__' + s('banner_pos')
    df['campaign_x_device'] = s('ad_campaign_id') + '__' + s('device_type')
    return df


# ── 3. Bayesian CTR + log-frequency statistics ────────────────────────────────
def _compute_ctr_stats(df):
    gctr  = float(df[LABEL].astype(int).mean())
    stats = {'__gctr__': gctr}
    for col in STAT_COLS:
        if col not in df.columns:
            continue
        keys = df[col].fillna('__NULL__').astype(str).values
        ys   = df[LABEL].astype(int).values
        tmp  = pd.DataFrame({'k': keys, 'y': ys})
        grp  = tmp.groupby('k')['y'].agg(cnt='count', total='sum')
        stats[f'{col}__ctr'] = (
            (grp['total'].astype(float) + gctr * SMOOTH_K)
            / (grp['cnt'] + SMOOTH_K)
        ).to_dict()
        stats[f'{col}__cnt'] = np.log1p(grp['cnt'].astype(float)).to_dict()
    return stats


def _apply_ctr_stats(df, stats):
    gctr = stats.get('__gctr__', BASE_CTR)
    for col in STAT_COLS:
        if col not in df.columns:
            continue
        keys = df[col].fillna('__NULL__').astype(str)
        df[f'{col}_ctr'] = keys.map(stats.get(f'{col}__ctr', {})).fillna(gctr)
        df[f'{col}_cnt'] = keys.map(stats.get(f'{col}__cnt', {})).fillna(0.0)
    return df


# ── 4. Label encoding (unseen values -> -1) ───────────────────────────────────
def _fit_encoders(df, cols):
    enc = {}
    for col in cols:
        if col not in df.columns:
            continue
        vals = df[col].fillna('__NULL__').astype(str)
        enc[col] = {v: i for i, v in enumerate(sorted(vals.unique()))}
    return enc


def _apply_encoders(df, cols, enc):
    for col in cols:
        if col not in df.columns or col not in enc:
            continue
        vals    = df[col].fillna('__NULL__').astype(str)
        df[col] = vals.map(enc[col]).fillna(-1).astype(np.int32)
    return df


# ── 5. Special master pipeline ────────────────────────────────────────────────
def _make_features_special(df, ctr_stats=None, encoders=None, is_train=True):
    df = df.copy()

    df = _add_temporal(df)                          # drops 'timestamp'

    if 'is_app' in df.columns:
        df['is_app'] = df['is_app'].astype(int)     # bool -> 0/1

    df = _add_interactions(df)                      # new string columns

    if is_train:
        ctr_stats = _compute_ctr_stats(df)          # build from training data only
    df = _apply_ctr_stats(df, ctr_stats)            # add _ctr and _cnt columns

    all_cat = (OBJ_CAT_COLS + INT_AS_CAT +
               ['time_of_day', 'pub_x_is_app', 'pub_x_device',
                'site_cat_x_device', 'banner_x_device',
                'creative_x_banner', 'campaign_x_device'])
    if is_train:
        encoders = _fit_encoders(df, all_cat)
    df = _apply_encoders(df, all_cat, encoders)     # cat -> int32

    if LABEL in df.columns:
        df[LABEL] = df[LABEL].astype(int)           # bool -> 0/1

    df = df.drop(columns=[c for c in _SPECIAL_DROP if c in df.columns])

    return df, ctr_stats, encoders


# ── Legacy variants (simple / standard / advanced) ────────────────────────────
def transform_pipeline(
    tr: pd.DataFrame,
    val: pd.DataFrame,
    te: pd.DataFrame,
    variant: str,
    label_col: str
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Applies temporal, interaction, and Bayesian features based on variant."""

    # --- Global Configs ---
    _SMOOTH_K = 10
    _BASE_CTR = tr[label_col].mean()
    _STAT_COLS = [
        'publisher_id', 'ad_id', 'ad_campaign_id', 'user_id',
        'site_id', 'app_id', 'site_category',
        'device_type', 'device_model', 'device_conn_type',
        'creative_size', 'banner_pos', 'hour', 'day_of_week',
        'C1', 'C15', 'C16', 'C21', 'user_segment', 'time_of_day',
        'pub_x_is_app', 'pub_x_device', 'site_cat_x_device', 'campaign_x_device',
    ]
    CAT_COLS = [
        "day_of_week", "device_type", "device_model", "device_conn_type",
        "site_id", "site_domain", "site_category",
        "app_id", "app_domain",
        "ad_id", "ad_campaign_id", "publisher_id", "creative_size",
        "user_segment", "C1", "C15", "C16", "C21",
    ]

    # --- 1. Temporal & Interaction Features ---
    if variant in ["standard", "advanced"]:
        for df in [tr, val, te]:
            df['is_weekend'] = df['day_of_week'].isin(['Sat', 'Sun']).astype(int)
            df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
            df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
            cond = [(df['hour'] >= 6) & (df['hour'] < 12),
                    (df['hour'] >= 12) & (df['hour'] < 17),
                    (df['hour'] >= 17) & (df['hour'] < 22)]
            df['time_of_day'] = np.select(cond, ['morning', 'afternoon', 'evening'], default='night')
            s = lambda c: df[c].fillna('__NULL__').astype(str)
            df['pub_x_is_app'] = s('publisher_id') + '__' + s('is_app')
            df['pub_x_device'] = s('publisher_id') + '__' + s('device_type')
            df['site_cat_x_device'] = s('site_category') + '__' + s('device_type')
            if variant == "advanced":
                df['banner_x_device'] = s('banner_pos') + '__' + s('device_type')
                df['creative_x_banner'] = s('creative_size') + '__' + s('banner_pos')
                df['campaign_x_device'] = s('ad_campaign_id') + '__' + s('device_type')

    # --- 2. Bayesian CTR & Log-Frequency (Leakage-free) ---
    if variant in ["standard", "advanced"]:
        active_stat_cols = [c for c in _STAT_COLS if c in tr.columns]
        for col in active_stat_cols:
            keys = tr[col].fillna('__NULL__').astype(str)
            ys = tr[label_col].astype(int)
            grp = pd.DataFrame({'k': keys, 'y': ys}).groupby('k')['y'].agg(cnt='count', total='sum')
            ctr_map = ((grp['total'] + _BASE_CTR * _SMOOTH_K) / (grp['cnt'] + _SMOOTH_K)).to_dict()
            cnt_map = np.log1p(grp['cnt'].astype(float)).to_dict()
            for df in [tr, val, te]:
                k_df = df[col].fillna('__NULL__').astype(str)
                df[f'{col}_ctr'] = k_df.map(ctr_map).fillna(_BASE_CTR)
                df[f'{col}_cnt'] = k_df.map(cnt_map).fillna(0.0)

    # --- 3. Label Encoding ---
    all_cats = list(set(CAT_COLS + ['time_of_day', 'pub_x_is_app', 'pub_x_device',
                                    'site_cat_x_device', 'banner_x_device',
                                    'creative_x_banner', 'campaign_x_device']))
    for col in all_cats:
        if col in tr.columns:
            le = LabelEncoder()
            combined = pd.concat([tr[col], val[col], te[col]]).astype(str)
            le.fit(combined)
            tr[col] = le.transform(tr[col].astype(str))
            val[col] = le.transform(val[col].astype(str))
            te[col] = le.transform(te[col].astype(str))

    return tr, val, te


TARGET    = "clicked"
DROP_COLS = ["impression_id", "timestamp", TARGET]


def build_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    tr_mask: pd.Series,
    val_mask: pd.Series,
    variant: str = "special",
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[str]]:
    """Slices data, applies transformations, and returns training-ready arrays."""

    tr  = train_df[tr_mask].copy()
    val = train_df[val_mask].copy()
    te  = test_df.copy()

    if variant == "special":
        # ── Special variant: uses _make_features_special ──────────────────────
        # Drop timestamp before processing (temporal fn expects it present)
        tr_f,  ctr_stats, encoders = _make_features_special(tr,  is_train=True)
        val_f, _,         _        = _make_features_special(val, ctr_stats=ctr_stats, encoders=encoders, is_train=False)
        te_f,  _,         _        = _make_features_special(te,  ctr_stats=ctr_stats, encoders=encoders, is_train=False)

        feature_cols = [c for c in tr_f.columns if c != TARGET]
        X_tr  = tr_f[feature_cols].values.astype(float)
        y_tr  = tr_f[TARGET].values
        X_val = val_f[feature_cols].values.astype(float)
        y_val = val_f[TARGET].values
        X_te  = te_f[feature_cols].values.astype(float)
    else:
        # ── Legacy variants: simple / standard / advanced ─────────────────────
        tr, val, te = transform_pipeline(tr, val, te, variant, label_col=TARGET)
        internal_drop = ["user_id"]
        feature_cols = [c for c in tr.columns if c not in DROP_COLS + internal_drop]
        X_tr  = tr[feature_cols].values.astype(float)
        y_tr  = tr[TARGET].values
        X_val = val[feature_cols].values.astype(float)
        y_val = val[TARGET].values
        X_te  = te[feature_cols].values.astype(float)

    return X_tr, y_tr, X_val, y_val, X_te, feature_cols


print('Feature engineering pipeline ready (variants: simple | standard | advanced | special).')

In [ ]:
X_tr, y_tr, X_val, y_val, X_te, feature_cols = build_features(
    train_df=train,
    test_df=test,
    tr_mask=mask_tr,
    val_mask=mask_val,
    variant="special",
)
print(f"X_tr: {X_tr.shape}  |  X_val: {X_val.shape}  |  X_te: {X_te.shape}")
print(f"features ({len(feature_cols)}): {feature_cols}")

In [ ]:
def get_imbalance_ratio(y_tr: np.ndarray) -> float:
    """Calculates the ratio of negative to positive samples."""
    num_neg = np.sum(y_tr == 0)
    num_pos = np.sum(y_tr == 1)
    return num_neg / num_pos

ratio = get_imbalance_ratio(y_tr)
print(f"Imbalance Ratio: {ratio:.2f}")

BASELINE_PARAMS = {
    "objective":        "binary:logistic",
    "eval_metric":      "logloss",
    "learning_rate":    0.05,
    "max_depth":        6,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "n_estimators":     2000,
    "early_stopping_rounds": 100,
    "verbosity":        0,
    "scale_pos_weight": ratio,
}

baseline = xgb.XGBClassifier(**BASELINE_PARAMS)
baseline.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

val_preds_baseline = baseline.predict_proba(X_val)[:, 1]
print(f"baseline  NCE: {nce(y_val, val_preds_baseline):.4f}  |  logloss: {log_loss(y_val, val_preds_baseline):.4f}")

In [ ]:
results = {}

for variant in ["simple", "standard", "advanced", "special"]:
    X_tr_v, y_tr_v, X_val_v, y_val_v, _, cols_v = build_features(
        train, test, mask_tr, mask_val, variant=variant
    )
    m = xgb.XGBClassifier(**BASELINE_PARAMS)
    m.fit(X_tr_v, y_tr_v, eval_set=[(X_val_v, y_val_v)], verbose=False)
    preds = m.predict_proba(X_val_v)[:, 1]
    results[variant] = {
        "nce":     nce(y_val_v, preds),
        "logloss": log_loss(y_val_v, preds),
        "n_feats": len(cols_v),
    }
    print(f"{variant:10s}  NCE={results[variant]['nce']:.4f}  logloss={results[variant]['logloss']:.4f}  features={results[variant]['n_feats']}")

# advanced should now beat standard (previously 0.9992 vs 0.9950 due to rolling CTR bug)
best = min(results, key=lambda k: results[k]["nce"])
print(f"\nbest variant: {best}")

## 6 · Baseline XGBoost

In [ ]:
importance = pd.Series(baseline.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importance.to_string())

# Percentile gate: drop bottom 10%
threshold = importance.quantile(0.05)
keep_cols = importance[importance > threshold].index.tolist()

# Manually drop known decoys regardless of whether they survived the percentile gate
KNOWN_DECOYS = {"ad_quality_score", "user_depth", "C1", "C15", "C21"}
keep_cols = [c for c in keep_cols if c not in KNOWN_DECOYS]
drop_cols = [c for c in feature_cols if c not in keep_cols]

print(f"\nkeeping {len(keep_cols)} / {len(feature_cols)} features  (dropped: {drop_cols})")

In [ ]:
keep_idx = [feature_cols.index(c) for c in keep_cols]

X_tr_k  = X_tr[:, keep_idx]
X_val_k = X_val[:, keep_idx]
X_te_k  = X_te[:, keep_idx]

pruned = xgb.XGBClassifier(**BASELINE_PARAMS)
pruned.fit(X_tr_k, y_tr, eval_set=[(X_val_k, y_val)], verbose=False)

val_preds_pruned = pruned.predict_proba(X_val_k)[:, 1]
print(f"pruned    NCE: {nce(y_val, val_preds_pruned):.4f}  |  logloss: {log_loss(y_val, val_preds_pruned):.4f}")
print(f"baseline  NCE: {nce(y_val, val_preds_baseline):.4f}  (reference)")

## 8 · Hyperparameter Tuning (Optuna)

In [ ]:
def optuna_objective(trial: optuna.Trial) -> float:
    params = {
        "objective":             "binary:logistic",
        "eval_metric":           "logloss",
        "verbosity":             0,
        "learning_rate":         trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth":             trial.suggest_int("max_depth", 4, 10),
        "min_child_weight":      trial.suggest_int("min_child_weight", 10, 100),
        "subsample":             trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":      trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":             trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda":            trial.suggest_float("reg_lambda", 0.0, 5.0),
        "n_estimators":          500,
        "early_stopping_rounds": 100,
        "scale_pos_weight": ratio,
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr_k, y_tr, eval_set=[(X_val_k, y_val)], verbose=False)
    # Store best iteration so §9 can use it instead of falling back to 500
    trial.set_user_attr("best_iteration", model.best_iteration)
    return log_loss(y_val, model.predict_proba(X_val_k)[:, 1])


study = optuna.create_study(direction="minimize")
study.optimize(optuna_objective, n_trials=100, show_progress_bar=True)

print(f"\nbest logloss : {study.best_value:.4f}")
print(f"best params  : {study.best_params}")
print(f"best iter    : {study.best_trial.user_attrs.get('best_iteration')}")

## 9 · Train Final Model on Full Train Set

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression

TRAINING_CTR = train["clicked"].mean()  # 0.3069 — fixed anchor for NCE denominator

# ── Rebuild features on full train (no val) to maximise training data ─────────
X_tr_full, y_tr_full, _, _, X_te_full, _ = build_features(
    train, test,
    tr_mask=pd.Series([True]  * len(train), index=train.index),
    val_mask=pd.Series([False] * len(train), index=train.index),
)
X_tr_full_k = X_tr_full[:, keep_idx]
X_te_full_k = X_te_full[:, keep_idx]

# ── Tuned model on fold — source of val predictions for calibration ───────────
tuned_model = xgb.XGBClassifier(**{**BASELINE_PARAMS, **study.best_params})
tuned_model.fit(X_tr_k, y_tr, eval_set=[(X_val_k, y_val)], verbose=False)
val_proba = tuned_model.predict_proba(X_val_k)[:, 1]

print(f"tuned val  NCE: {nce(y_val, val_proba):.6f}  |  logloss: {log_loss(y_val, val_proba):.6f}")
print(f"best iteration: {tuned_model.best_iteration}")

# ── Isotonic calibration fitted on val predictions ────────────────────────────
# Same model -> same score distribution -> calibration is valid for test
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(val_proba, y_val)
val_proba_cal = calibrator.predict(val_proba)

nce_raw = nce(y_val, val_proba)
nce_cal = nce(y_val, val_proba_cal)
ll_raw  = log_loss(y_val, val_proba)
ll_cal  = log_loss(y_val, val_proba_cal)

print(f'\n  Naïve baseline    NCE = 1.000000')
print(f'  Kaggle baseline   NCE = 0.923764')
print(f'  Adv. baseline     NCE = 0.859638  <- target')
print(f'  Tuned (raw)       NCE = {nce_raw:.6f}  log_loss={ll_raw:.6f}')
print(f'  Tuned (cal)       NCE = {nce_cal:.6f}  log_loss={ll_cal:.6f}')
delta = nce_cal - 0.859638
if delta < 0:
    print(f'  BEAT advanced baseline by {abs(delta):.6f}!')
else:
    print(f'  Gap to advanced baseline: {delta:.6f}')

# ── Reliability diagram ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (proba, title, color) in zip(axes, [
    (val_proba,     'Before Calibration', 'darkorange'),
    (val_proba_cal, 'After Calibration',  'steelblue'),
]):
    frac, mean_pred = calibration_curve(y_val, proba, n_bins=20, strategy='uniform')
    ax.plot(mean_pred, frac, 's-', lw=2, color=color, label='Model')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
    nv = nce(y_val, proba)
    ax.set_title(f'{title}\nNCE = {nv:.4f}', fontweight='bold')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.legend()
plt.suptitle('Reliability Diagram (Validation: Week 3)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Final model on full train, calibrated test predictions ────────────────────
n_estimators = tuned_model.best_iteration
best_params = {
    "objective":    "binary:logistic",
    "eval_metric":  "logloss",
    "verbosity":    0,
    "n_estimators": n_estimators,
    "scale_pos_weight": ratio,
    **study.best_params,
}
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_tr_full_k, y_tr_full, verbose=False)

test_proba_raw = final_model.predict_proba(X_te_full_k)[:, 1]
test_proba_cal = np.clip(calibrator.predict(test_proba_raw), 1e-6, 1 - 1e-6)

print(f'\nTest prediction statistics:')
print(f'  min  = {test_proba_cal.min():.4f}')
print(f'  max  = {test_proba_cal.max():.4f}')
print(f'  mean = {test_proba_cal.mean():.4f}  (train BASE_CTR = {TRAINING_CTR:.4f})')
print(f'  std  = {test_proba_cal.std():.4f}')

## 10 · Calibration

Isotonic regression fitted on val predictions from the tuned model.
Applied to test predictions from the same model (same distribution).
No logit shift — isotonic output clipped to [1e-6, 1-1e-6] for numerical safety.

## 11 · Submit

In [ ]:
submission = pd.DataFrame({
    "impression_id": test["impression_id"],
    "clicked":       test_proba_cal,
})

out_path = SUB_DIR / "submission.csv"
submission.to_csv(out_path, index=False)

assert len(submission) == len(test), "Row count mismatch!"
assert submission["clicked"].between(0, 1).all(), "Probabilities out of [0, 1]!"

print(f"saved → {out_path}  ({len(submission):,} rows)")
print(submission.head(5).to_string(index=False))

## 12 · Prediction Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Prediction Distribution", fontsize=13)

# ── 1. Test predictions histogram ────────────────────────────────────────────
ax = axes[0]
ax.hist(test_proba_cal, bins=80, color="#5b8db8", edgecolor="none", alpha=0.85)
ax.axvline(test_proba_cal.mean(), color="#e07b54", linewidth=1.5,
           label=f"mean={test_proba_cal.mean():.4f}")
ax.axvline(TRAINING_CTR, color="black", linewidth=1.2, linestyle="--",
           label=f"train CTR={TRAINING_CTR:.4f}")
ax.set_title("Test predictions")
ax.set_xlabel("Predicted CTR")
ax.set_ylabel("Count")
ax.legend(fontsize=8)

# ── 2. Val: raw vs calibrated ─────────────────────────────────────────────────
ax = axes[1]
ax.hist(val_proba,     bins=80, color="#5b8db8", edgecolor="none", alpha=0.6, label="raw val")
ax.hist(val_proba_cal, bins=80, color="#e07b54", edgecolor="none", alpha=0.6, label="calibrated val")
ax.axvline(TRAINING_CTR, color="black", linewidth=1.2, linestyle="--",
           label=f"train CTR={TRAINING_CTR:.4f}")
ax.set_title("Val: raw vs calibrated")
ax.set_xlabel("Predicted CTR")
ax.set_ylabel("Count")
ax.legend(fontsize=8)

# ── 3. Test predictions by decile ────────────────────────────────────────────
ax = axes[2]
deciles = pd.qcut(test_proba_cal, q=10, labels=False)
decile_means = pd.Series(test_proba_cal).groupby(deciles).mean()
ax.bar(range(1, 11), decile_means.values, color="#5b8db8", edgecolor="none")
ax.axhline(TRAINING_CTR, color="#e07b54", linewidth=1.2, linestyle="--",
           label=f"train CTR={TRAINING_CTR:.4f}")
ax.set_title("Mean predicted CTR by decile")
ax.set_xlabel("Decile (1=lowest, 10=highest)")
ax.set_ylabel("Mean predicted CTR")
ax.set_xticks(range(1, 11))
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nTest prediction stats:")
print(pd.Series(test_proba_cal).describe().to_string())